# Ollama on a Colab GPU - fully automatic

1. **Runtime -> Change runtime type -> T4 GPU.**
2. **Run the single cell below** (or *Runtime -> Run all*). That's it.

The cell installs Ollama, downloads the model, opens a tunnel and **announces its own URL**
to your local app (via your private `TOPIC`). Nothing to copy into `.env`; `python autopilot.py`
on your computer finds it by itself. The cell then keeps running and re-announces every 5 minutes.

*One-time setup: open the key icon (Secrets) in the left sidebar, add a secret named `NTFY_TOPIC` with the same value as `NTFY_TOPIC` in your local `.env`, and turn on "Notebook access". Keep it private - anyone who knows it could point your agent at their server.*

In [ ]:
MODEL = "qwen3:8b"   # fits a free T4. Try "qwen3:4b" (faster) or "qwen3:14b" (bigger GPUs)
from google.colab import userdata
TOPIC = userdata.get("NTFY_TOPIC")   # Colab Secrets (key icon, left sidebar); same value as NTFY_TOPIC in your local .env

import json, os, re, subprocess, time, urllib.request

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

# 1) GPU + Ollama
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
sh("apt-get install -y -qq zstd > /dev/null")
sh("curl -fsSL https://ollama.com/install.sh | sh")

server = subprocess.Popen(["ollama", "serve"],
                          env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434", "OLLAMA_KEEP_ALIVE": "-1"},
                          stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start; see ollama.log")

# 2) Model
sh(f"ollama pull {MODEL}")

# 3) Public tunnel (Cloudflare quick tunnel - no account needed)
sh("wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared")
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:11434", "--http-host-header", "localhost:11434"],
    stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
public_url = None
for _ in range(90):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        public_url = m.group(0)
        break
    time.sleep(1)
if not public_url:
    raise RuntimeError("Tunnel did not start; see tunnel.log")

# 4) Announce the URL to your local app, then keep the runtime alive
def announce():
    body = json.dumps({"url": public_url + "/api/generate", "model": MODEL, "ts": time.time()}).encode()
    urllib.request.urlopen(urllib.request.Request(f"https://ntfy.sh/{TOPIC}", data=body), timeout=15)

print("Ready:", public_url)
while True:
    try:
        announce()
        print(time.strftime("%H:%M:%S"), "announced - local app will pick this up automatically", flush=True)
    except Exception as exc:
        print("could not announce:", exc, flush=True)
    time.sleep(300)